<a href="https://colab.research.google.com/github/EreAguilaReal/Proyecto_IA/blob/modelo/modelo_adeudos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Priorización de adeudos: parte de modelo
Cada celda corresponde a un paso del plan (días 1 a 5). Córrelas en orden en Google Colab.

## 1. Cargar los datos

In [ ]:
import pandas as pd, numpy as np
from google.colab import files

subido = files.upload()          # sube aquí tu Excel o CSV
nombre = list(subido.keys())[0]
if nombre.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(nombre)
else:
    df = pd.read_csv(nombre, encoding='utf-8-sig', sep=None, engine='python')
print(df.shape)

Saving adeudos_fiscales_400_personas.xlsx to adeudos_fiscales_400_personas.xlsx
(400, 9)


## 2. Arreglar los acentos (FÃ­sica → Física)

In [ ]:
def arreglar(s):
    for enc in ('cp1252', 'latin-1'):
        try:
            return s.encode(enc).decode('utf-8')
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return s

for c in df.select_dtypes('object'):
    df[c] = df[c].map(lambda x: arreglar(x) if isinstance(x, str) else x)

print(df['Tipo_Persona'].unique())
print(df['Estatus_Actual'].unique())

['Física' 'Moral']
['Exigible' 'En Litigio' 'Convenio de Pago' 'No Localizado']


## 3. Revisión rápida

In [ ]:
print(df.head())
print(df.dtypes)
print('Nulos por columna:\n', df.isna().sum())
print('IDs duplicados:', df.duplicated('ID_Registro').sum())
print(df['Estatus_Actual'].value_counts(normalize=True).round(3))
print(df[['Monto_Original_MXN', 'Antiguedad_Dias']].describe().round(1))

  ID_Registro RFC_Anonimizado Tipo_Persona Entidad_Federativa  \
0    REG-0001   OKHU761019WKK       Física             Puebla   
1    REG-0002    DHX720621BX3        Moral            Jalisco   
2    REG-0003   UALZ91091701P       Física         Nuevo León   
3    REG-0004   OSLW890903GUI       Física             Edomex   
4    REG-0005   DYNR950926BT1       Física         Nuevo León   

            Concepto_Adeudo  Monto_Original_MXN Fecha_Notificacion  \
0          Crédito Exigible            39777.35         2026-05-21   
1          Auditoría Fiscal            26180.40         2025-09-23   
2  Autocorrección Pendiente           205242.50         2026-03-10   
3          Auditoría Fiscal             2571.64         2022-12-27   
4             Crédito Firme            18863.59         2025-11-22   

   Antiguedad_Dias    Estatus_Actual  
0              173          Exigible  
1              413        En Litigio  
2              245        En Litigio  
3             1414          Exig

## 4. ¿Antigüedad y fecha dicen lo mismo?

In [ ]:
df['Fecha_Notificacion'] = pd.to_datetime(df['Fecha_Notificacion'], dayfirst=True)
ref = (df['Fecha_Notificacion'] + pd.to_timedelta(df['Antiguedad_Dias'], unit='D')).mode()[0]
print('Fecha de referencia implícita en Antiguedad_Dias:', ref.date())
# Si todas las filas dan la misma fecha, las dos columnas son redundantes: usa solo Antiguedad_Dias.

Fecha de referencia implícita en Antiguedad_Dias: 2026-11-10


## 5. ¿El estatus depende de las demás variables?
Si los porcentajes son casi iguales en todas las filas, el estatus se generó al azar y ningún modelo podrá aprender nada.

In [ ]:
for col in ['Tipo_Persona', 'Entidad_Federativa', 'Concepto_Adeudo']:
    print(pd.crosstab(df[col], df['Estatus_Actual'], normalize='index').round(2), '\n')
print(df.groupby('Estatus_Actual')[['Monto_Original_MXN', 'Antiguedad_Dias']].median().round(0))

Estatus_Actual  Convenio de Pago  En Litigio  Exigible  No Localizado
Tipo_Persona                                                         
Física                      0.21        0.19      0.52           0.09
Moral                       0.14        0.22      0.52           0.13 

Estatus_Actual      Convenio de Pago  En Litigio  Exigible  No Localizado
Entidad_Federativa                                                       
Baja California                 0.13        0.30      0.48           0.09
CDMX                            0.20        0.22      0.48           0.10
Edomex                          0.11        0.16      0.66           0.08
Guanajuato                      0.25        0.13      0.48           0.13
Jalisco                         0.16        0.18      0.62           0.04
Nuevo León                      0.18        0.20      0.53           0.10
Puebla                          0.22        0.18      0.55           0.04
Veracruz                        0.22        0.19    

## 6. Naive Bayes y métricas (exactitud, precisión, recall, F1)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score

# Variables: ID y RFC no entran. Monto y antigüedad se agrupan en 5 rangos.
X = df[['Tipo_Persona', 'Entidad_Federativa', 'Concepto_Adeudo']].copy()
X['monto_rango'] = pd.qcut(df['Monto_Original_MXN'], 5, labels=False, duplicates='drop')
X['antig_rango'] = pd.qcut(df['Antiguedad_Dias'], 5, labels=False, duplicates='drop')
y = df['Estatus_Actual']

X_enc = OrdinalEncoder().fit_transform(X).astype(int)
min_cat = X_enc.max(axis=0) + 1
modelo = CategoricalNB(alpha=1.0, min_categories=min_cat)   # alpha=1 = suavizado de Laplace

Xtr, Xte, ytr, yte = train_test_split(X_enc, y, test_size=0.2, stratify=y, random_state=42)
modelo.fit(Xtr, ytr)
pred = modelo.predict(Xte)

print('Exactitud Naive Bayes:', round(accuracy_score(yte, pred), 3))
base = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
print('Exactitud de siempre predecir la clase mayoritaria:', round(accuracy_score(yte, base.predict(Xte)), 3))
print(classification_report(yte, pred, zero_division=0))
print(pd.crosstab(yte, pred, rownames=['real'], colnames=['predicho']))

cv = StratifiedKFold(5, shuffle=True, random_state=42)
print('F1 macro por pliegue:', cross_val_score(modelo, X_enc, y, cv=cv, scoring='f1_macro').round(3))

Exactitud Naive Bayes: 0.475
Exactitud de siempre predecir la clase mayoritaria: 0.512
                  precision    recall  f1-score   support

Convenio de Pago       0.00      0.00      0.00        15
      En Litigio       0.00      0.00      0.00        16
        Exigible       0.49      0.93      0.64        41
   No Localizado       0.00      0.00      0.00         8

        accuracy                           0.47        80
       macro avg       0.12      0.23      0.16        80
    weighted avg       0.25      0.47      0.33        80

predicho          Convenio de Pago  En Litigio  Exigible
real                                                    
Convenio de Pago                 0           0        15
En Litigio                       0           0        16
Exigible                         2           1        38
No Localizado                    0           0         8
F1 macro por pliegue: [0.189 0.2   0.158 0.194 0.216]


## 7. Puntaje de prioridad y comparación con los baselines

In [ ]:
modelo.fit(X_enc, y)
proba = pd.DataFrame(modelo.predict_proba(X_enc), columns=modelo.classes_, index=df.index)

# SUPUESTO (validar con el equipo): "cobro viable" = Exigible + Convenio de Pago
df['p_viable'] = proba['Exigible'] + proba['Convenio de Pago']
df['prioridad'] = df['p_viable'] * df['Monto_Original_MXN']

# Baselines del pitch: solo monto y solo antigüedad
df['rank_prioridad'] = df['prioridad'].rank(ascending=False)
df['rank_monto'] = df['Monto_Original_MXN'].rank(ascending=False)
df['rank_antig'] = df['Antiguedad_Dias'].rank(ascending=False)
print(df[['rank_prioridad', 'rank_monto', 'rank_antig']].corr(method='spearman').round(2))

top = df.sort_values('prioridad', ascending=False).head(20)
print(top[['ID_Registro', 'Concepto_Adeudo', 'Monto_Original_MXN', 'Antiguedad_Dias',
           'Estatus_Actual', 'p_viable', 'prioridad']].round(2))

                rank_prioridad  rank_monto  rank_antig
rank_prioridad            1.00        0.99       -0.02
rank_monto                0.99        1.00       -0.01
rank_antig               -0.02       -0.01        1.00
    ID_Registro           Concepto_Adeudo  Monto_Original_MXN  \
47     REG-0048             Crédito Firme          1003671.70   
178    REG-0179          Auditoría Fiscal           856188.23   
263    REG-0264             Crédito Firme           868084.33   
300    REG-0301         Multa por Omisión           793245.82   
138    REG-0139          Auditoría Fiscal           682487.37   
84     REG-0085             Crédito Firme           778888.82   
89     REG-0090             Crédito Firme           559099.30   
313    REG-0314             Crédito Firme           679970.85   
329    REG-0330         Multa por Omisión           462629.48   
103    REG-0104  Autocorrección Pendiente           487384.61   
238    REG-0239             Crédito Firme           453344.56   
